# Bayit Dashboard — Richmond BC Synagogue Site Finder

Interactive map dashboard for identifying properties in Richmond, BC that are zoned to permit Religious Assembly.

**Data sources:**
- ParcelMap BC (128k parcels via WFS)
- City of Richmond Bylaw 8500 (561 assembly-zoned properties)
- OpenStreetMap (parks, schools, hospitals — excluded areas)
- BC ALR boundary
- TransLink GTFS transit stops

## Quick Start
1. Run **Cell 1** (Setup) to install packages
2. Run **Cell 2** (Load Data) — downloads parquet files from GitHub automatically
3. Run **Cell 3** (Dashboard) to launch the interactive map

To refresh data from original sources, use the **Data Refresh** section at the bottom.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 1: SETUP — Install packages (run once per session)
# ═══════════════════════════════════════════════════════════════

!pip install -q geopandas lonboard ipywidgets pyarrow requests

import warnings
warnings.filterwarnings('ignore')
print('Packages installed')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 2: LOAD DATA — Download from GitHub and load
# ═══════════════════════════════════════════════════════════════

import geopandas as gpd
import pandas as pd
import numpy as np
from pathlib import Path
import requests
import os

# ─── Configuration ───
GITHUB_REPO = 'accubotai/bayit-notebook'
GITHUB_BRANCH = 'main'
DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)

DATA_FILES = [
    'parcels.parquet',
    'assembly_candidates.parquet',
    'alr_boundary.parquet',
    'excluded_areas.parquet',
    'transit_stops.parquet',
]

# Detect environment
try:
    from google.colab import files as colab_files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# ─── Download data from GitHub if not cached locally ───
def download_data():
    """Download parquet files from GitHub repo."""
    base_url = f'https://github.com/{GITHUB_REPO}/raw/{GITHUB_BRANCH}/data'
    for filename in DATA_FILES:
        local_path = DATA_DIR / filename
        if local_path.exists():
            print(f'  [cached] {filename}')
            continue
        url = f'{base_url}/{filename}'
        print(f'  Downloading {filename}...', end=' ')
        resp = requests.get(url, timeout=60, allow_redirects=True)
        resp.raise_for_status()
        local_path.write_bytes(resp.content)
        print(f'{len(resp.content) / 1024 / 1024:.1f} MB')

print('Downloading data from GitHub...')
download_data()

# ─── Load all datasets ───
def load_data():
    """Load all datasets from parquet files and enrich assembly candidates."""
    data = {}
    for name in ['parcels', 'assembly_candidates', 'alr_boundary', 'excluded_areas', 'transit_stops']:
        key = name.replace('_candidates', '').replace('_boundary', '').replace('_areas', '').replace('_stops', '')
        data[name] = gpd.read_parquet(DATA_DIR / f'{name}.parquet')
        print(f'  Loaded {name}: {len(data[name]):,} rows')

    parcels = data['parcels']
    assembly = data['assembly_candidates']

    # Enrich: join matched assembly candidates with parcel owner/area info
    matched = assembly[assembly['matched_parcel_id'].notna()].copy()
    if len(matched) > 0:
        parcel_info = parcels[['id', 'owner_type', 'lot_area_sqm', 'owner_name']].copy()
        parcel_info = parcel_info.rename(columns={'id': 'matched_parcel_id'})
        matched = matched.merge(parcel_info, on='matched_parcel_id', how='left', suffixes=('', '_parcel'))

    unmatched = assembly[assembly['matched_parcel_id'].isna()].copy()
    for col in ['owner_type', 'lot_area_sqm', 'owner_name']:
        if col not in unmatched.columns:
            unmatched[col] = None

    enriched = pd.concat([matched, unmatched], ignore_index=True)
    enriched = gpd.GeoDataFrame(enriched, geometry='geom')

    # Compute ALR status for matched parcels
    alr_union = data['alr_boundary'].union_all()
    matched_idx = enriched['matched_parcel_id'].notna()
    if matched_idx.any():
        parcel_geoms = parcels.set_index('id')['geom']
        enriched.loc[matched_idx, 'in_alr'] = (
            enriched.loc[matched_idx, 'matched_parcel_id']
            .astype(int)
            .map(parcel_geoms)
            .apply(lambda g: g.intersects(alr_union) if g is not None else False)
            .values
        )
    enriched['in_alr'] = enriched['in_alr'].fillna(False)

    data['assembly_enriched'] = enriched
    print(f'\nAll data loaded. {len(enriched):,} assembly candidates ({matched_idx.sum()} with parcel match).')
    return data

data = load_data()

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 3: INTERACTIVE DASHBOARD — Map + Filters
# ═══════════════════════════════════════════════════════════════

import lonboard
from lonboard import Map, PolygonLayer, ScatterplotLayer
import ipywidgets as widgets
from IPython.display import display, clear_output

# ─── Constants ───
RICHMOND_CENTER = (49.166, -123.137)
ZONE_OPTIONS = ['All zones', 'ASY', 'CA', 'CDT1', 'CC', 'CEA']
OWNER_OPTIONS = ['All owners', 'Local Government', 'Private', 'Crown Agency', 'Federal', 'Unclassified']
OCCUPIED_TYPES = {
    'apartments', 'house', 'residential', 'retail', 'restaurant', 'hotel',
    'commercial', 'school', 'cafe', 'supermarket', 'kindergarten',
    'place_of_worship', 'hospital', 'clinic', 'library',
}

# ─── Color parcels by owner type ───
def get_parcel_colors(gdf):
    colors = np.full((len(gdf), 4), [180, 180, 180, 140], dtype=np.uint8)
    colors[gdf['owner_type'] == 'Municipal', :] = [59, 130, 246, 160]
    colors[gdf['owner_type'] == 'Local Government', :] = [59, 130, 246, 160]
    colors[gdf['owner_type'] == 'Crown Provincial', :] = [139, 92, 246, 160]
    return colors

# ─── Filter widgets ───
w_zone = widgets.Dropdown(options=ZONE_OPTIONS, value='All zones', description='Zoning:')
w_owner = widgets.Dropdown(options=OWNER_OPTIONS, value='All owners', description='Owner:')
w_min_area = widgets.IntText(value=1000, description='Min area m\u00b2:', style={'description_width': 'initial'})
w_max_area = widgets.IntText(value=50000, description='Max area m\u00b2:', style={'description_width': 'initial'})
w_exclude_alr = widgets.Checkbox(value=True, description='Exclude ALR', style={'description_width': 'initial'})
w_exclude_occupied = widgets.Checkbox(value=False, description='Exclude occupied', style={'description_width': 'initial'})
w_only_matched = widgets.Checkbox(value=False, description='Only confirmed parcels', style={'description_width': 'initial'})
w_show_base = widgets.Checkbox(value=True, description='Show all parcels (base layer)', style={'description_width': 'initial'})
w_status = widgets.HTML(value='<i>Initializing...</i>')

assembly = data['assembly_enriched'].copy()
parcels = data['parcels'].copy()

def apply_filters():
    """Apply all filters and return filtered assembly GeoDataFrame."""
    df = assembly.copy()
    if w_zone.value != 'All zones':
        df = df[df['zoning'].str.contains(w_zone.value, na=False)]
    if w_owner.value != 'All owners':
        df = df[df['owner_type'] == w_owner.value]
    if w_min_area.value:
        df = df[~((df['lot_area_sqm'].notna()) & (df['lot_area_sqm'] < w_min_area.value))]
    if w_max_area.value:
        df = df[~((df['lot_area_sqm'].notna()) & (df['lot_area_sqm'] > w_max_area.value))]
    if w_exclude_alr.value:
        df = df[~df['in_alr'].astype(bool)]
    if w_exclude_occupied.value:
        df = df[~df['place_type'].isin(OCCUPIED_TYPES)]
    if w_only_matched.value:
        df = df[df['matched_parcel_id'].notna()]
    return df

def get_assembly_layers(filtered):
    matched_ids = filtered[filtered['matched_parcel_id'].notna()]['matched_parcel_id'].astype(int).tolist()
    polys = parcels[parcels['id'].isin(matched_ids)].copy()
    points = filtered[filtered['matched_parcel_id'].isna()].copy()
    return polys, points

map_output = widgets.Output()

def render_map(change=None):
    """Re-render the map with current filters."""
    filtered = apply_filters()
    assembly_polys, assembly_points = get_assembly_layers(filtered)
    layers = []

    if w_show_base.value and len(parcels) > 0:
        layers.append(PolygonLayer.from_geopandas(
            parcels, get_fill_color=get_parcel_colors(parcels),
            get_line_color=[30, 41, 59, 200], line_width_min_pixels=0.5,
            auto_highlight=True, pickable=True,
        ))

    if len(assembly_polys) > 0:
        layers.append(PolygonLayer.from_geopandas(
            assembly_polys, get_fill_color=[245, 158, 11, 160],
            get_line_color=[180, 83, 9, 220], line_width_min_pixels=2,
            auto_highlight=True, pickable=True,
        ))

    if len(assembly_points) > 0:
        layers.append(ScatterplotLayer.from_geopandas(
            assembly_points, get_fill_color=[245, 158, 11, 200],
            get_line_color=[180, 83, 9, 255], get_radius=40,
            line_width_min_pixels=2, pickable=True,
        ))

    m = Map(layers=layers, _initial_view_state={
        'latitude': RICHMOND_CENTER[0], 'longitude': RICHMOND_CENTER[1],
        'zoom': 13, 'pitch': 0,
    }, basemap_style=lonboard.basemap.CartoBasemap.Voyager)

    w_status.value = (
        f'<b style="color:#b45309">{len(filtered)}</b> of {len(assembly)} assembly parcels shown '
        f'({len(assembly_polys)} polygons, {len(assembly_points)} points) '
        f'| Base layer: {len(parcels):,} parcels'
    )
    with map_output:
        clear_output(wait=True)
        display(m)

for w in [w_zone, w_owner, w_min_area, w_max_area, w_exclude_alr, w_exclude_occupied, w_only_matched, w_show_base]:
    w.observe(render_map, names='value')

filter_box = widgets.VBox([
    widgets.HTML('<h3 style="color:#92400e; margin:0">\U0001f3db Assembly Parcel Filters</h3>'),
    widgets.HBox([w_zone, w_owner]),
    widgets.HBox([w_min_area, w_max_area]),
    widgets.HBox([w_exclude_alr, w_exclude_occupied, w_only_matched]),
    w_show_base, w_status,
], layout=widgets.Layout(padding='10px', border='2px solid #f59e0b', border_radius='8px', margin='0 0 10px 0'))

display(filter_box)
display(map_output)
render_map()

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 4: VIEW FILTERED TABLE
# ═══════════════════════════════════════════════════════════════

filtered = apply_filters()
display_cols = ['address', 'zoning', 'owner_type', 'lot_area_sqm', 'place_type', 'place_name', 'in_alr', 'matched_pid']
available_cols = [c for c in display_cols if c in filtered.columns]
print(f'{len(filtered)} assembly parcels match current filters:\n')
filtered[available_cols].sort_values('lot_area_sqm', ascending=False, na_position='last')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 5: EXPORT SHORTLIST TO CSV
# ═══════════════════════════════════════════════════════════════

filtered = apply_filters()
export_cols = ['address', 'zoning', 'owner_type', 'lot_area_sqm', 'place_type', 'place_name', 'in_alr', 'matched_pid', 'lat', 'lng']
available_cols = [c for c in export_cols if c in filtered.columns]
export_path = DATA_DIR / 'shortlist.csv'
filtered[available_cols].to_csv(export_path, index=False)
print(f'Exported {len(filtered)} parcels to {export_path}')
if IN_COLAB:
    colab_files.download(str(export_path))

---

# Data Refresh

Run these cells to update data from original sources. After refreshing, re-run **Cell 2** and **Cell 3**.

To make refreshed data permanent, commit the updated parquet files back to the GitHub repo.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# REFRESH A: Parcels from ParcelMap BC WFS (~5-10 min)
# ═══════════════════════════════════════════════════════════════

import time
from shapely.geometry import shape

WFS_URL = 'https://openmaps.gov.bc.ca/geo/pub/ows'
LAYER = 'pub:WHSE_CADASTRE.PMBC_PARCEL_FABRIC_POLY_SVW'
MUNICIPALITY = 'Richmond, City of'
BBOX = {'xmin': -123.30, 'ymin': 49.08, 'xmax': -123.00, 'ymax': 49.23}
GRID_COLS, GRID_ROWS = 10, 5

dx = (BBOX['xmax'] - BBOX['xmin']) / GRID_COLS
dy = (BBOX['ymax'] - BBOX['ymin']) / GRID_ROWS
all_features, total_tiles = [], GRID_COLS * GRID_ROWS

for row in range(GRID_ROWS):
    for col in range(GRID_COLS):
        tile = row * GRID_COLS + col + 1
        xmin, ymin = BBOX['xmin'] + col * dx, BBOX['ymin'] + row * dy
        cql = f"MUNICIPALITY='{MUNICIPALITY}' AND BBOX(SHAPE,{xmin},{ymin},{xmin+dx},{ymin+dy},'EPSG:4326')"
        try:
            resp = requests.get(WFS_URL, params={
                'service': 'WFS', 'version': '2.0.0', 'request': 'GetFeature',
                'typeName': LAYER, 'outputFormat': 'application/json',
                'srsName': 'EPSG:4326', 'CQL_FILTER': cql, 'count': 10000,
            }, timeout=60)
            all_features.extend(resp.json().get('features', []))
            print(f'  Tile {tile}/{total_tiles}: {len(all_features)} total', end='\r')
        except Exception as e:
            print(f'  Tile {tile}: ERROR {e}')
        time.sleep(0.5)

seen, rows = set(), []
for f in all_features:
    pid = f['properties'].get('PID')
    if pid in seen: continue
    seen.add(pid)
    rows.append({'pid': pid, 'owner_type': f['properties'].get('OWNER_TYPE', 'Private'),
        'lot_area_sqm': f['properties'].get('FEATURE_AREA_SQM', 0),
        'civic_address': None, 'owner_name': None, 'source': 'parcelmap_bc', 'geom': shape(f['geometry'])})

gdf = gpd.GeoDataFrame(rows, geometry='geom', crs='EPSG:4326')
gdf['id'] = range(1, len(gdf) + 1)
gdf.to_parquet(DATA_DIR / 'parcels.parquet')
print(f'\nSaved {len(gdf):,} parcels')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# REFRESH B: Geocode assembly CSV (~10 min)
# ═══════════════════════════════════════════════════════════════

import csv, re
from shapely.geometry import Point

csv_path = DATA_DIR / 'Properties zoned to permit Religious Assembly.csv'
if not csv_path.exists():
    # Download from GitHub repo
    url = f'https://github.com/{GITHUB_REPO}/raw/{GITHUB_BRANCH}/data/Properties%20zoned%20to%20permit%20Religious%20Assembly.csv'
    csv_path.write_bytes(requests.get(url, timeout=30).content)

with open(csv_path) as f:
    raw = list(csv.DictReader(f))
addr_zones = {}
for r in raw:
    addr_zones.setdefault(r['Address'].strip(), set()).add(r['Zoning'].strip())
entries = [{'address': a, 'zoning': ', '.join(sorted(z))} for a, z in addr_zones.items()]
print(f'{len(raw)} rows -> {len(entries)} unique addresses')

def geocode(address):
    variants = [address]
    m = re.search(r'\bNo\s+(\d+)\s+Rd\b', address)
    if m: variants.append(re.sub(r'\bNo\s+\d+\s+Rd\b', f'Number {m.group(1)} Road', address))
    for v in variants:
        try:
            resp = requests.get('https://nominatim.openstreetmap.org/search', params={
                'street': v, 'city': 'Richmond', 'state': 'British Columbia',
                'country': 'Canada', 'format': 'json', 'limit': 1,
                'viewbox': '-123.30,49.08,-123.00,49.23', 'bounded': 1,
            }, headers={'User-Agent': 'BayitDashboard/2.0'}, timeout=10)
            if resp.ok and resp.json():
                r = resp.json()[0]
                return float(r['lat']), float(r['lon']), r.get('type', ''), r.get('display_name', '').split(',')[0]
        except: pass
        time.sleep(1.1)
    return None, None, '', ''

results = []
for i, e in enumerate(entries):
    lat, lng, pt, pn = geocode(e['address'])
    results.append({**e, 'lat': lat, 'lng': lng, 'place_type': pt, 'place_name': pn})
    print(f'  [{i+1}/{len(entries)}] {e["address"]}', end='\r')
    time.sleep(1.1)

df = pd.DataFrame(results)
geoms = [Point(r['lng'], r['lat']) if pd.notna(r['lat']) else None for _, r in df.iterrows()]
gdf = gpd.GeoDataFrame(df, geometry=geoms, crs='EPSG:4326').rename(columns={'geometry': 'geom'}).set_geometry('geom')
gdf['id'] = range(1, len(gdf) + 1)
gdf['matched_parcel_id'] = gdf['matched_pid'] = gdf['notes'] = None

# Spatial match
parcels_for_match = gpd.read_parquet(DATA_DIR / 'parcels.parquet')
valid = gdf[gdf['geom'].notna()].copy()
joined = gpd.sjoin(valid, parcels_for_match[['id', 'pid', 'geom']], how='left', predicate='within')
gdf.loc[joined.index, 'matched_parcel_id'] = joined['id_right']
gdf.loc[joined.index, 'matched_pid'] = joined.get('pid_right', joined.get('pid'))
gdf.to_parquet(DATA_DIR / 'assembly_candidates.parquet')
print(f'\nSaved {len(gdf)} candidates ({gdf["matched_parcel_id"].notna().sum()} matched)')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# REFRESH C: ALR boundary from BC WFS
# ═══════════════════════════════════════════════════════════════

resp = requests.get('https://openmaps.gov.bc.ca/geo/pub/ows', params={
    'service': 'WFS', 'version': '2.0.0', 'request': 'GetFeature',
    'typeName': 'pub:WHSE_LEGAL_ADMIN_BOUNDARIES.OATS_ALR_POLYS',
    'outputFormat': 'application/json', 'srsName': 'EPSG:4326',
    'CQL_FILTER': "BBOX(SHAPE,-123.30,49.08,-123.00,49.23,'EPSG:4326')",
}, timeout=60)
alr = gpd.GeoDataFrame.from_features(resp.json()['features'], crs='EPSG:4326')
alr = alr.rename(columns={'geometry': 'geom'}).set_geometry('geom')
alr['id'] = range(1, len(alr) + 1)
alr[['id', 'geom']].to_parquet(DATA_DIR / 'alr_boundary.parquet')
print(f'Saved {len(alr)} ALR polygons')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# REFRESH D: Excluded areas from OSM Overpass
# ═══════════════════════════════════════════════════════════════

from shapely.geometry import Polygon, Point
BB = '49.08,-123.30,49.23,-123.00'
query = f"""[out:json][timeout:120];(
  way["leisure"="park"]({BB});way["amenity"="school"]({BB});
  way["amenity"="hospital"]({BB});way["amenity"="place_of_worship"]({BB});
  way["amenity"="fire_station"]({BB});way["amenity"="police"]({BB});
  way["amenity"="kindergarten"]({BB});way["amenity"="childcare"]({BB});
  way["landuse"="cemetery"]({BB});
  node["amenity"="school"]({BB});node["amenity"="hospital"]({BB});
  node["amenity"="fire_station"]({BB});node["amenity"="police"]({BB});
);out body;>;out skel qt;"""
resp = requests.post('https://overpass-api.de/api/interpreter', data={'data': query}, timeout=180)
nodes = {e['id']: (e['lon'], e['lat']) for e in resp.json()['elements'] if e['type'] == 'node'}
geoms = []
for el in resp.json()['elements']:
    if el['type'] == 'way' and 'nodes' in el:
        coords = [nodes[n] for n in el['nodes'] if n in nodes]
        if len(coords) >= 3:
            try: geoms.append(Polygon(coords))
            except: pass
    elif el['type'] == 'node' and 'tags' in el:
        geoms.append(Point(el['lon'], el['lat']).buffer(0.0003))
gdf = gpd.GeoDataFrame({'id': range(1, len(geoms)+1)}, geometry=geoms, crs='EPSG:4326')
gdf = gdf.rename(columns={'geometry': 'geom'}).set_geometry('geom')
gdf.to_parquet(DATA_DIR / 'excluded_areas.parquet')
print(f'Saved {len(gdf)} excluded areas')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# REFRESH E: TransLink GTFS transit stops
# ═══════════════════════════════════════════════════════════════

import zipfile, io
from shapely.geometry import Point
resp = requests.get('https://gtfs-static.translink.ca/gtfs/google_transit.zip', timeout=60)
with zipfile.ZipFile(io.BytesIO(resp.content)) as z:
    stops = pd.read_csv(z.open('stops.txt'))
richmond = stops[(stops['stop_lat'] >= 49.08) & (stops['stop_lat'] <= 49.23) &
                  (stops['stop_lon'] >= -123.30) & (stops['stop_lon'] <= -123.00)].copy()
geoms = [Point(r['stop_lon'], r['stop_lat']) for _, r in richmond.iterrows()]
gdf = gpd.GeoDataFrame(richmond, geometry=geoms, crs='EPSG:4326')
gdf = gdf.rename(columns={'stop_lat': 'lat', 'stop_lon': 'lng', 'geometry': 'geom'}).set_geometry('geom')
gdf['id'] = range(1, len(gdf) + 1)
gdf['route_type'] = 3
gdf[['id', 'stop_name', 'route_type', 'lat', 'lng', 'geom']].to_parquet(DATA_DIR / 'transit_stops.parquet')
print(f'Saved {len(gdf)} transit stops')

In [ ]:
# After refreshing, re-run Cell 2 and Cell 3 to see updated data.
print('Data refresh complete! Re-run Cell 2 and Cell 3 to reload.')